In [1]:
"""
═══════════════════════════════════════════════════════════════════════════════
UNIVERSAL AFib DETECTION MODEL TRAINER
Version: 5.0 (Final - Production Ready)
═══════════════════════════════════════════════════════════════════════════════

PURPOSE:
    Train any AFib detection model with research-paper-accurate preprocessing
    
SUPPORTED MODELS:
    1. CNN-BiLSTM (Andersen 2019) - RR intervals
    2. CNN-LSTM-Focal (Petmezas 2021) - Raw beats with Focal Loss
    3. AFib-ResLSTM (Novel) - Raw ECG with attention
    4. ResNet-BiLSTM-Attention (Jia 2020) - Raw ECG with attention
    5. Lightweight-ResNet (Lueken 2025) - Sparse model
    
WORKFLOW:
    Load Data → Split → Create DataLoaders → Initialize Model → Train → Evaluate → Save
═══════════════════════════════════════════════════════════════════════════════
"""


'\n═══════════════════════════════════════════════════════════════════════════════\nUNIVERSAL AFib DETECTION MODEL TRAINER\nVersion: 5.0 (Final - Production Ready)\n═══════════════════════════════════════════════════════════════════════════════\n\nPURPOSE:\n    Train any AFib detection model with research-paper-accurate preprocessing\n\nSUPPORTED MODELS:\n    1. CNN-BiLSTM (Andersen 2019) - RR intervals\n    2. CNN-LSTM-Focal (Petmezas 2021) - Raw beats with Focal Loss\n    3. AFib-ResLSTM (Novel) - Raw ECG with attention\n    4. ResNet-BiLSTM-Attention (Jia 2020) - Raw ECG with attention\n    5. Lightweight-ResNet (Lueken 2025) - Sparse model\n\nWORKFLOW:\n    Load Data → Split → Create DataLoaders → Initialize Model → Train → Evaluate → Save\n═══════════════════════════════════════════════════════════════════════════════\n'

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
import json
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    roc_auc_score, 
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

PYTORCH DATASET

In [ ]:
class ECGDataset(Dataset):
    """
    Universal PyTorch Dataset for ECG data.
    
    HANDLES:
        - RR intervals: (N, 30)
        - Raw beats: (N, 187) 
        - Raw ECG windows: (N, 7500) or (N, 6000)
    
    AUTOMATIC FEATURES:
        Label conversion (strings → integers)
        Handles any data shape
        No manual preprocessing needed
    """
    def __init__(self, X, y):
        """
        Args:
            X (numpy.ndarray): Features array
            y (numpy.ndarray): Labels array (can be int, str, or mixed)
        """
        #convert features to pytorch tensor
        self.X = torch.FloatTensor(X)
        # Handles: integers, strings, mixed types
        if y.dtype == object or not np.issubdtype(y.dtype, np.integer):
            #Label mapping dictionary
            label_map = {
                'N': 0, 'A': 1,           # CINC17 format
                'normal': 0, 'afib': 1,   # Text format
                'Normal': 0, 'AFib': 1,   # Title case
                '0': 0, '1': 1            # String numbers
            }
            y_int =[]
            for label in y:
                try:
                    # try direct integer conversion first 
                    y_int.append(int(label))
                except(ValueError ,TypeError):
                    y_int.append(label_map.get(str(label).lower(), 0))
            y = np.array(y_int, dtype= np.int64)
        else:
            y = y.astype(np.int64)
        self.y = torch.LongTensor(y)

    def __len__(self):
        """Return total number of samples"""
        return len(self.X)
    
    def __getitem__(self, idx):
        """
        Get a single sample.
        
        Returns:
            tuple: (ecg_data, label)
                - ecg_data: Shape varies by model
                - label: Single integer (0 or 1)
        """
        return self.X[idx] , self.y[idx]


MAIN TRAINING FUNCTION

In [ ]:
def train_afib_model(
        # REQUIRED ARGUMENTS
        model_name : str,# e.g., 'cnn_bilstm'
        model_class, # The PyTorch nn.Module class
        model_config: dict, # Configuration dict from model file

        # PATH ARGUMENTS (Always use absolute paths!)
        processed_data_dir: str,     
        results_save_dir: str,        

        # TRAINING HYPERPARAMETERS
        epochs : int = 100,
        learning_rate : float = 0.001,
        batch_size : int =32,

        # DATA SPLIT RATIOS
        test_size: float = 0.15,      # 15% for final testing
        val_size: float = 0.15,       # 15% for validation

        # OTHER SETTINGS
        device: str = 'auto',         # 'auto', 'cuda', or 'cpu'
        early_stopping_patience: int = 15,
        use_class_weights: bool = True,
        random_seed: int = 42,):
    """
        ═══════════════════════════════════════════════════════════════════════════
        UNIVERSAL AFIB MODEL TRAINER
        ═══════════════════════════════════════════════════════════════════════════
        
        This function handles the complete training pipeline:
        
        STEP 1: Load preprocessed data (X_processed.npy, y_processed.npy)
        STEP 2: Split into train/val/test sets (stratified)
        STEP 3: Create PyTorch DataLoaders
        STEP 4: Initialize model, loss function, optimizer
        STEP 5: Training loop with early stopping
        STEP 6: Evaluation on test set
        STEP 7: Save model, results, and plots
        
        ═══════════════════════════════════════════════════════════════════════════
        
        Args:
            model_name: Name matching preprocessed folder (e.g., 'cnn_bilstm')
            model_class: The actual PyTorch model class (not instantiated)
            model_config: Dictionary with model settings
            processed_data_dir: Root folder containing processed/{model_name}/
            results_save_dir: Root folder to save results/{model_name}/
            ... (other args documented above)
        
        Returns:
            dict: Complete results including metrics and training history
        
        Example:
            >>> from cnn_bilstm import MODEL_CLASS, MODEL_CONFIG
            >>> results = train_afib_model(
            ...     model_name='cnn_bilstm',
            ...     model_class=MODEL_CLASS,
            ...     model_config=MODEL_CONFIG,
            ...     processed_data_dir='/path/to/data/processed',
            ...     results_save_dir='/path/to/results'
            ... )
        """

    # Set random seeds for reproducibility
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)
    if(torch.cuda.is_available()):
        torch.cuda.manual_seed_all(random_seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    
    # Determine compute device
    if device == 'auto':
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    else:
        device = torch.device(device)

    # Print configuration
    print(f"CONFIGURATION")
    print(f"{'─'*80}")
    print(f"   Model:          {model_name}")
    print(f"   Device:         {device}")
    print(f"   Epochs:         {epochs}")
    print(f"   Learning Rate:  {learning_rate}")
    print(f"   Batch Size:     {batch_size}")
    print(f"   Random Seed:    {random_seed}")
    print(f"   Early Stop:     {early_stopping_patience} epochs patience")

    # Convert paths to Path objects for safety
    processed_data_dir = Path(processed_data_dir).resolve()
    results_save_dir = Path(results_save_dir).resolve()
    
    # Model-specific directories
    model_data_path = processed_data_dir / model_name
    model_results_path = results_save_dir / model_name
    
    print(f"\n📂 PATHS")
    print(f"{'─'*80}")
    print(f"   Data:    {model_data_path}")
    print(f"   Results: {model_results_path}")

    # STEP 1: LOAD PREPROCESSED DATA
    X_path = model_data_path/ 'X_processed.npy'
    y_path = model_data_path/ 'y_processed.npy'

    if not (X_path.exists()):
        raise FileNotFoundError(
            f"\n{'='*80}\n"
            f"ERROR: Preprocessed data not found!\n"
            f"{'='*80}\n"
            f"Looking for: {X_path}\n\n"
            f"SOLUTION: Run preprocessing first:\n"
            f"  1. Import: from research_accurate_preprocessor import ResearchAccuratePreprocessor\n"
            f"  2. Run: preprocessor.preprocess_for_model('{model_name}')\n"
            f"{'='*80}\n"
        )
    # Load numpy arrays
    print(f"Loading data from: {model_data_path.name}/")
    X = np.load(X_path)
    y = np.load(y_path)
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")

    # Analyze class distribution
    unique_labels, counts = np.unique(y, return_counts=True)
    print(f"\nCLASS DISTRIBUTION")
    print(f"   {'─'*40}")
    for label, count in zip(unique_labels, counts):
        label_name = 'Normal' if str(label) == '0' else 'AFib'
        percentage = (count / len(y)) * 100
        bar = '█' * int(percentage / 2)  # Visual bar
        print(f"      {label_name:8s} ({label}): {count:6,} samples ({percentage:5.1f}%) {bar}")

    #validate data
    if len(unique_labels) <2:
        raise ValueError(f"\n{'='*80}\n"
            f"ERROR: Only one class found in data!\n"
            f"{'='*80}\n"
            f"Found classes: {unique_labels}\n"
            f"Cannot train a binary classifier with only one class.\n"
            f"Check your preprocessing labels.\n"
            f"{'='*80}\n")
    
    # STEP 2: SPLIT DATA (Train/Val/Test)
    try:
        X_trainval , X_test , y_trainval , y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state= random_seed,
            stratify= y  # Maintains class distribution
        )
        # Second split: Separate validation from training
        # Adjust val_size to be relative to remaining data
        val_size_adjusted = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval, y_trainval,
            test_size=val_size_adjusted,
            random_state=random_seed,
            stratify=y_trainval
        )
        
        print(f"Stratified split successful")
    except ValueError as e:
        # If stratification fails (too few samples), do random split
        print(f"Stratification failed: {e}")
        print(f"Using random split (no stratification)")
        
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_seed
        )
        val_size_adjusted = val_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval, y_trainval, 
            test_size=val_size_adjusted, 
            random_state=random_seed
        )

    # STEP 3: CREATE PYTORCH DATALOADERS

    # Create dataset objects
    train_dataset = ECGDataset(X_train ,y_train)
    val_dataset = ECGDataset(X_val, y_val)
    test_dataset = ECGDataset(X_test, y_test)

    # Create dataloaders
    # num_workers=0 is safest for debugging
    # Increase to 2-4 for faster training (if no errors)
    num_workers = 0
    pin_memory = (device.type == 'cuda')  # Speed up GPU transfer
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,  # Shuffle training data each epoch
        num_workers=num_workers,
        pin_memory=pin_memory
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,  # Don't shuffle validation
        num_workers=num_workers,
        pin_memory=pin_memory
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,  # Don't shuffle test
        num_workers=num_workers,
        pin_memory=pin_memory
    )
    
    print(f"      DataLoaders created:")
    print(f"      Train: {len(train_loader):4d} batches × {batch_size} samples")
    print(f"      Val:   {len(val_loader):4d} batches × {batch_size} samples")
    print(f"      Test:  {len(test_loader):4d} batches × {batch_size} samples")

    # STEP 4: INITIALIZE MODEL, LOSS, OPTIMIZER

    config = model_config.copy()
    config['learning_rate'] = learning_rate
    config['batch_size'] = batch_size
    config['epochs'] = epochs

    # create model instance and move to device 
    model = model_class(config).to(device)

    #count model parameters 
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # SETUP LOSS FUNCTION
    try:
        # Try importing LOSS_FUNCTION from model module
        import sys
        import importlib.util
        
        # Get the model's module
        model_module = sys.modules[model_class.__module__]
        
        if hasattr(model_module, 'LOSS_FUNCTION'):
            # Use custom loss function
            criterion = model_module.LOSS_FUNCTION()
            print(f"      Using custom loss: {criterion.__class__.__name__}")
            if hasattr(criterion, 'gamma'):
                print(f"      Focal Loss gamma: {criterion.gamma}")
            if hasattr(criterion, 'alpha'):
                print(f"      Focal Loss alpha: {criterion.alpha}")
        else:
            raise AttributeError("No custom loss")
            
    except (AttributeError, KeyError):
        # Use standard CrossEntropy with optional class weights
        if use_class_weights:
            # Calculate inverse frequency weights
            class_counts = np.bincount(y_train)
            
            if len(class_counts) == 2 and 0 not in class_counts:
                # Both classes present - calculate weights
                total = len(y_train)
                weights = total / (len(class_counts) * class_counts)
                class_weights = torch.FloatTensor(weights).to(device)
                
                print(f"      Using weighted CrossEntropyLoss")
                print(f"      Class weights: Normal={class_weights[0]:.3f}, "
                    f"AFib={class_weights[1]:.3f}")
                criterion = nn.CrossEntropyLoss(weight=class_weights)
            else:
                # Something wrong with class distribution
                print(f"      Cannot compute class weights (counts: {class_counts})")
                print(f"      Using standard CrossEntropyLoss")
                criterion = nn.CrossEntropyLoss()
        else:
            criterion = nn.CrossEntropyLoss()
            print(f"      Using standard CrossEntropyLoss")
    # SETUP OPTIMIZER
    # Get weight decay from config if available
    weight_decay = config.get('optimizer', {}).get('weight_decay', 0)
    
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr=learning_rate,
        weight_decay=weight_decay
    )

    print(f"      Type:         Adam")
    print(f"      Learning rate: {learning_rate}")
    if weight_decay > 0:
        print(f"      Weight decay:  {weight_decay}")

    # SETUP LEARNING RATE SCHEDULER
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',      # Maximize validation AUROC
    factor=0.5,      # Reduce LR by half
    patience=5,      # Wait 5 epochs before reducing
    verbose=True,
    min_lr=1e-7      # Don't reduce below this
)
    
    print(f"\n  LEARNING RATE SCHEDULER")
    print(f"   {'─'*60}")
    print(f"      Type:     ReduceLROnPlateau")
    print(f"      Metric:   Validation AUROC (maximize)")
    print(f"      Factor:   0.5 (halve LR)")
    print(f"      Patience: 5 epochs")

    # STEP 5: TRAINING LOOP
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'val_f1': [],
        'val_auroc': [],
        'learning_rates': []
    }
    
    best_val_auroc = -1.0          # Track best validation AUROC
    best_model_state = None         # Store best model weights
    patience_counter = 0            # Early stopping counter

    for epoch in range(1, epochs + 1):
        # TRAINING PHASE
       
        model.train()  # Set model to training mode (enables dropout, etc.)
        
        train_loss = 0.0
        train_preds = []
        train_labels = []
        
        # Progress bar for this epoch
        pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{epochs} [Train]')
        
        for batch_ecg, batch_labels in pbar:
           
            # Move data to device (CPU/GPU)
            batch_ecg = batch_ecg.to(device)
            batch_labels = batch_labels.to(device)
            
             
            # Forward pass
            optimizer.zero_grad()  # Clear gradients from last step
            
            outputs = model(batch_ecg)
            
            # Handle models that return tuple (logits, attention_weights)
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs
            
             
            # Calculate loss
            loss = criterion(logits, batch_labels)
            
             
            # Backward pass
            loss.backward()  # Calculate gradients
            
            # Gradient clipping (prevents exploding gradients)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
             
            # Update weights
            optimizer.step()
            
             
            # Track metrics
            train_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(batch_labels.cpu().numpy())
            
            # Update progress bar
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        # Calculate epoch training metrics
        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = accuracy_score(train_labels, train_preds)

        # VALIDATION PHASE
        model.eval()  # Set model to evaluation mode (disables dropout)
        
        val_loss = 0.0
        val_preds = []
        val_labels = []
        val_probs = []
        
        with torch.no_grad():  # Don't calculate gradients (saves memory)
            for batch_ecg, batch_labels in tqdm(val_loader, 
                                                 desc=f'Epoch {epoch}/{epochs} [Val]  ',
                                                 leave=False):
                 
                # Move data to device
                batch_ecg = batch_ecg.to(device)
                batch_labels = batch_labels.to(device)
                
                 
                # Forward pass (no gradients needed)
                outputs = model(batch_ecg)
                if isinstance(outputs, tuple):
                    logits = outputs[0]
                else:
                    logits = outputs
                
                 
                # Calculate loss
                loss = criterion(logits, batch_labels)
                val_loss += loss.item()
                
                 
                # Get predictions and probabilities
                probs = F.softmax(logits, dim=1)
                preds = torch.argmax(probs, dim=1)
                
                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(batch_labels.cpu().numpy())
                
                # Store probability of AFib class (class 1) for AUROC
                if probs.shape[1] > 1:
                    val_probs.extend(probs[:, 1].cpu().numpy())
                else:
                    val_probs.extend(torch.zeros_like(preds).cpu().numpy())
        
        # Calculate epoch validation metrics
        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = accuracy_score(val_labels, val_preds)
        epoch_val_f1 = f1_score(val_labels, val_preds, zero_division=0)
        
        # Calculate AUROC (handle edge case of only one class)
        epoch_val_auroc = 0.5  # Default if calculation fails
        try:
            if len(np.unique(val_labels)) > 1:
                epoch_val_auroc = roc_auc_score(val_labels, val_probs)
            else:
                print(f"      ⚠️  Only one class in validation set")
        except Exception as e:
            print(f"      ⚠️  AUROC error: {e}")
        
         
        # Store history
         
        history['train_loss'].append(float(epoch_train_loss))
        history['train_acc'].append(float(epoch_train_acc))
        history['val_loss'].append(float(epoch_val_loss))
        history['val_acc'].append(float(epoch_val_acc))
        history['val_f1'].append(float(epoch_val_f1))
        history['val_auroc'].append(float(epoch_val_auroc))
        history['learning_rates'].append(float(optimizer.param_groups[0]['lr']))
        
         
        # Print epoch results
        print(f"\n   {'─'*70}")
        print(f"   Epoch {epoch}/{epochs} Results:")
        print(f"   {'─'*70}")
        print(f"   Train │ Loss: {epoch_train_loss:7.4f} │ Acc: {epoch_train_acc:6.4f}")
        print(f"   Val   │ Loss: {epoch_val_loss:7.4f} │ Acc: {epoch_val_acc:6.4f} │ "
              f"F1: {epoch_val_f1:6.4f} │ AUROC: {epoch_val_auroc:6.4f}")
        
         
        # Learning rate scheduling
        scheduler.step(epoch_val_auroc)
        # Early stopping check
         
        if epoch_val_auroc > best_val_auroc:
            # New best model!
            best_val_auroc = epoch_val_auroc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"   ✅ New best model! (AUROC: {best_val_auroc:.4f})")
        else:
            # No improvement
            patience_counter += 1
            print(f"   ⏳ No improvement ({patience_counter}/{early_stopping_patience})")
        
        # Check if we should stop early
        if patience_counter >= early_stopping_patience:
            print(f"\n   ⏹️  Early stopping triggered at epoch {epoch}")
            break
    # STEP 6: FINAL TEST EVALUATION

    # Load best model
    if best_model_state is None:
        print("   No best model saved (training may have failed)")
        print("   Using last epoch weights")
        best_model_state = model.state_dict()
    
    model.load_state_dict(best_model_state)
    model.eval()
    
    # ───────────────────────────────────────────────────────────────────────
    # Evaluate on test set
    # ───────────────────────────────────────────────────────────────────────
    test_preds = []
    test_labels = []
    test_probs = []
    
    with torch.no_grad():
        for batch_ecg, batch_labels in tqdm(test_loader, desc='Testing'):
            batch_ecg = batch_ecg.to(device)
            batch_labels = batch_labels.to(device)
            
            outputs = model(batch_ecg)
            if isinstance(outputs, tuple):
                logits = outputs[0]
            else:
                logits = outputs
            
            probs = F.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)
            
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(batch_labels.cpu().numpy())
            
            if probs.shape[1] > 1:
                test_probs.extend(probs[:, 1].cpu().numpy())
            else:
                test_probs.extend(torch.zeros_like(preds).cpu().numpy())
    
    # ───────────────────────────────────────────────────────────────────────
    # Calculate test metrics
    # ───────────────────────────────────────────────────────────────────────
    test_acc = accuracy_score(test_labels, test_preds)
    test_f1 = f1_score(test_labels, test_preds, zero_division=0)
    
    test_auroc = 0.5
    try:
        if len(np.unique(test_labels)) > 1:
            test_auroc = roc_auc_score(test_labels, test_probs)
    except Exception as e:
        print(f"  Test AUROC error: {e}")
    
    # ───────────────────────────────────────────────────────────────────────
    # Confusion matrix and derived metrics
    # ───────────────────────────────────────────────────────────────────────
    cm = confusion_matrix(test_labels, test_preds)
    
    # Calculate sensitivity and specificity
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    else:
        print(f"   Unexpected confusion matrix shape: {cm.shape}")
        tn = fp = fn = tp = 0
        sensitivity = specificity = ppv = npv = 0.0
    
    # ───────────────────────────────────────────────────────────────────────
    # Print test results
    # ───────────────────────────────────────────────────────────────────────
    print(f"\n TEST SET PERFORMANCE")
    print(f"   {'═'*70}")
    print(f"   Accuracy:    {test_acc:.4f}  ({test_acc*100:.2f}%)")
    print(f"   F1-Score:    {test_f1:.4f}")
    print(f"   AUROC:       {test_auroc:.4f}")
    print(f"   Sensitivity: {sensitivity:.4f}  (Recall for AFib)")
    print(f"   Specificity: {specificity:.4f}  (Recall for Normal)")
    print(f"   PPV:         {ppv:.4f}  (Precision for AFib)")
    print(f"   NPV:         {npv:.4f}  (Precision for Normal)")
    
    print(f"\n  CONFUSION MATRIX")
    print(f"   {'─'*40}")
    print(f"              Predicted")
    print(f"            Normal    AFib")
    print(f"   Actual N  {tn:6d}  {fp:6d}")
    print(f"          A  {fn:6d}  {tp:6d}")
    
    # ═══════════════════════════════════════════════════════════════════════
    # STEP 7: SAVE RESULTS
    # ═══════════════════════════════════════════════════════════════════════
    
    print(f"\n{'═'*80}")
    print(f"STEP 7: SAVING RESULTS")
    print(f"{'═'*80}\n")
    
    # ───────────────────────────────────────────────────────────────────────
    # Create results directory
    # ───────────────────────────────────────────────────────────────────────
    model_results_path.mkdir(parents=True, exist_ok=True)
    
    # ───────────────────────────────────────────────────────────────────────
    # Save model checkpoint
    # ───────────────────────────────────────────────────────────────────────
    checkpoint = {
        'model_state_dict': best_model_state,
        'config': config,
        'history': history,
        'test_metrics': {
            'accuracy': float(test_acc),
            'f1_score': float(test_f1),
            'auroc': float(test_auroc),
            'sensitivity': float(sensitivity),
            'specificity': float(specificity),
            'ppv': float(ppv),
            'npv': float(npv),
            'confusion_matrix': cm.tolist()
        },
        'training_info': {
            'date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'random_seed': random_seed,
            'device': str(device),
            'total_epochs': len(history['train_loss']),
            'best_epoch': len(history['val_auroc']) - patience_counter - 1,
            'best_val_auroc': float(best_val_auroc)
        }
    }
    
    torch.save(checkpoint, model_results_path / 'best_model.pth')
    print(f"   Model checkpoint saved: best_model.pth")
    
    # ───────────────────────────────────────────────────────────────────────
    # Save results as JSON
    # ───────────────────────────────────────────────────────────────────────
    results = {
        'model_name': model_name,
        'config': config,
        'test_metrics': checkpoint['test_metrics'],
        'training_info': checkpoint['training_info'],
        'history': history
    }
    
    with open(model_results_path / 'results.json', 'w') as f:
        json.dump(results, f, indent=2)
    print(f"  Results saved: results.json")
    
    # ───────────────────────────────────────────────────────────────────────
    # Save training history plots
    # ───────────────────────────────────────────────────────────────────────
    _plot_training_history(history, model_results_path)
    print(f"   ✅ Training plots saved: training_history.png")
    
    # ───────────────────────────────────────────────────────────────────────
    # Save classification report
    # ───────────────────────────────────────────────────────────────────────
    report = classification_report(
        test_labels, test_preds, 
        target_names=['Normal', 'AFib'],
        digits=4
    )
    with open(model_results_path / 'classification_report.txt', 'w') as f:
        f.write(report)
    print(f"   Classification report saved: classification_report.txt")
    
    print(f"\n   All results saved to: {model_results_path}")
    
    # ═══════════════════════════════════════════════════════════════════════
    # FINAL SUMMARY
    # ═══════════════════════════════════════════════════════════════════════
    
    print(f"\n{'═'*80}")
    print(f"{'TRAINING COMPLETE!':^80}")
    print(f"{'═'*80}\n")
    
    print(f"     Final Test Performance:")
    print(f"      Accuracy:    {test_acc:.4f}")
    print(f"      F1-Score:    {test_f1:.4f}")
    print(f"      AUROC:       {test_auroc:.4f}")
    print(f"      Sensitivity: {sensitivity:.4f}")
    print(f"      Specificity: {specificity:.4f}")
    
    print(f"\n    Results Location: {model_results_path}")
    print(f"\n{'═'*80}\n")
    
    return results



In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# HELPER FUNCTION: Plot Training History
# ═══════════════════════════════════════════════════════════════════════════

def _plot_training_history(history, save_dir):
    """
    Create and save comprehensive training history plots.
    
    Creates a 2×2 grid showing:
        1. Training vs Validation Loss
        2. Training vs Validation Accuracy
        3. Validation F1 and AUROC
        4. Learning Rate Schedule
    
    Args:
        history (dict): Training history from training loop
        save_dir (Path): Directory to save the plot
    """
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Training History', fontsize=16, fontweight='bold')
    
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    # ───────────────────────────────────────────────────────────────────────
    # Plot 1: Loss
    # ───────────────────────────────────────────────────────────────────────
    axes[0, 0].plot(epochs_range, history['train_loss'], 
                    label='Train Loss', marker='o', linewidth=2)
    axes[0, 0].plot(epochs_range, history['val_loss'], 
                    label='Val Loss', marker='s', linewidth=2)
    axes[0, 0].set_xlabel('Epoch', fontsize=12)
    axes[0, 0].set_ylabel('Loss', fontsize=12)
    axes[0, 0].set_title('Training vs Validation Loss', fontsize=13, fontweight='bold')
    axes[0, 0].legend(fontsize=10)
    axes[0, 0].grid(True, alpha=0.3)
    
    # ───────────────────────────────────────────────────────────────────────
    # Plot 2: Accuracy
    # ───────────────────────────────────────────────────────────────────────
    axes[0, 1].plot(epochs_range, history['train_acc'], 
                    label='Train Acc', marker='o', linewidth=2)
    axes[0, 1].plot(epochs_range, history['val_acc'], 
                    label='Val Acc', marker='s', linewidth=2)
    axes[0, 1].set_xlabel('Epoch', fontsize=12)
    axes[0, 1].set_ylabel('Accuracy', fontsize=12)
    axes[0, 1].set_title('Training vs Validation Accuracy', fontsize=13, fontweight='bold')
    axes[0, 1].legend(fontsize=10)
    axes[0, 1].grid(True, alpha=0.3)
    
    # ───────────────────────────────────────────────────────────────────────
    # Plot 3: F1 and AUROC
    # ───────────────────────────────────────────────────────────────────────
    axes[1, 0].plot(epochs_range, history['val_f1'], 
                    label='Val F1', marker='o', color='green', linewidth=2)
    axes[1, 0].plot(epochs_range, history['val_auroc'], 
                    label='Val AUROC', marker='s', color='purple', linewidth=2)
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('Score', fontsize=12)
    axes[1, 0].set_title('Validation F1 and AUROC', fontsize=13, fontweight='bold')
    axes[1, 0].legend(fontsize=10)
    axes[1, 0].grid(True, alpha=0.3)
    
    # ───────────────────────────────────────────────────────────────────────
    # Plot 4: Learning Rate
    # ───────────────────────────────────────────────────────────────────────
    axes[1, 1].plot(epochs_range, history['learning_rates'], 
                    marker='o', color='red', linewidth=2)
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Learning Rate', fontsize=12)
    axes[1, 1].set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
    axes[1, 1].set_yscale('log')  # Log scale for LR
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_dir / 'training_history.png', dpi=300, bbox_inches='tight')
    plt.close()


# ═══════════════════════════════════════════════════════════════════════════
# COMMAND LINE INTERFACE (Optional)
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    print("═"*80)
    print("UNIVERSAL AFIB MODEL TRAINER")
    print("═"*80)
    print("\nThis script should be imported and used in a notebook or script.")
    print("\nExample usage:")
    print("""
    from simplified_trainer import train_afib_model
    from cnn_bilstm import MODEL_CLASS, MODEL_CONFIG
    
    results = train_afib_model(
        model_name='cnn_bilstm',
        model_class=MODEL_CLASS,
        model_config=MODEL_CONFIG,
        processed_data_dir='/path/to/data/processed',
        results_save_dir='/path/to/results',
        epochs=50,
        learning_rate=0.001,
        batch_size=32
    )
    """)
    print("═"*80)


════════════════════════════════════════════════════════════════════════════════
UNIVERSAL AFIB MODEL TRAINER
════════════════════════════════════════════════════════════════════════════════

This script should be imported and used in a notebook or script.

Example usage:

    from simplified_trainer import train_afib_model
    from cnn_bilstm import MODEL_CLASS, MODEL_CONFIG

    results = train_afib_model(
        model_name='cnn_bilstm',
        model_class=MODEL_CLASS,
        model_config=MODEL_CONFIG,
        processed_data_dir='/path/to/data/processed',
        results_save_dir='/path/to/results',
        epochs=50,
        learning_rate=0.001,
        batch_size=32
    )
    
════════════════════════════════════════════════════════════════════════════════
